# Hand history exploration (Parquet)

This notebook loads Parquet files produced by **`texas-holdem-parquet-export`** (see `docs/data-and-sqlite.md`).

**Setup** (from the repo root):

```bash
pip install -e '.[analysis]'
texas-holdem-parquet-export -o ./parquet_export
```

Point `EXPORT_DIR` below at your output folder (or copy the database and export elsewhere).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Directory containing hands.parquet, actions.parquet, …
EXPORT_DIR = Path("../parquet_export")

assert EXPORT_DIR.is_dir(), f"Run texas-holdem-parquet-export -o {EXPORT_DIR.resolve()} first"

## Load tables

In [ ]:
hands = pd.read_parquet(EXPORT_DIR / "hands.parquet")
actions = pd.read_parquet(EXPORT_DIR / "actions.parquet")
wide = pd.read_parquet(EXPORT_DIR / "actions_with_hands.parquet")

hands.head()

## Volume over time

Hand counts by calendar day (using `started_ms`).

In [ ]:
hands["started_dt"] = pd.to_datetime(hands["started_ms"], unit="ms", utc=True)
per_day = hands.set_index("started_dt").resample("1D").size()
per_day.plot(title="Hands per day", figsize=(9, 3))
plt.ylabel("count")
plt.tight_layout()

## Action mix

Uses `action_kind_label` from export (fold / call / raise / …).

In [ ]:
mix = actions["action_kind_label"].value_counts()
mix.plot(kind="bar", title="Actions by kind", figsize=(8, 3))
plt.tight_layout()

## Street distribution

Where decisions happened (preflop through river).

In [ ]:
street_order = ["preflop", "flop", "turn", "river", "showdown"]
sw = (
    actions["street_label"]
    .astype("category")
    .cat.set_categories(street_order, ordered=True)
)
sw.value_counts().sort_index().plot(kind="bar", title="Actions by street", figsize=(8, 3))
plt.tight_layout()

## Pot context on raises (wide table)

Example: average big-blind size when a raise occurs.

In [ ]:
raises = wide[wide["action_kind_label"] == "raise"]
if len(raises):
    print(raises[["bb_size", "seat", "street_label", "size_chips"]].head(10).to_string())
    print("Mean BB for raise rows:", raises["bb_size"].mean())
else:
    print("No raises in this export — play more hands or widen the sample.")